In [1]:
import os
import pandas as pd
import numpy as np

DATA_DIR = r"D:\L-NMPTDL&AI\Stage 1 + 2\DA-AI_Nhom8\DHHTTT20B_TCTA_Nhom8_Stage1\data"

books_path = os.path.join(DATA_DIR, "books_crawled.csv")
shopee_path = os.path.join(DATA_DIR, "shopee_data.csv")

print("Data folder:", DATA_DIR)


Data folder: D:\L-NMPTDL&AI\Stage 1 + 2\DA-AI_Nhom8\DHHTTT20B_TCTA_Nhom8_Stage1\data


In [2]:
books_df = pd.read_csv(books_path)
shopee_df = pd.read_csv(shopee_path)

print("Books shape:", books_df.shape)
print("Shopee shape:", shopee_df.shape)

print("\nBooks columns:")
print(books_df.columns.tolist())

print("\nShopee columns:")
print(shopee_df.columns.tolist())


Books shape: (300, 5)
Shopee shape: (20312, 20)

Books columns:
['title', 'price', 'rating', 'availability', 'url']

Shopee columns:
['price_ori', 'delivery', 'item_category_detail', 'specification', 'title', 'w_date', 'link_ori', 'item_rating', 'seller_name', 'idElastic', 'price_actual', 'sitename', 'idHash', 'total_rating', 'id', 'total_sold', 'pict_link', 'favorite', 'timestamp', 'desc']


In [3]:
# Chuẩn hóa dữ liệu Books to Scrape
books_static = pd.DataFrame({
    "product_id": [
        f"BOOK{i:06d}" for i in range(1, len(books_df) + 1)
    ],
    "title": books_df["title"].astype("string").str.strip(),
    "price": pd.to_numeric(
        books_df["price"].astype(str).str.replace(r"[^\d.]", "", regex=True),
        errors="coerce"
    ),
    "rating": pd.to_numeric(
        books_df["rating"].astype(str).str.extract(r"(\d+)", expand=False),
        errors="coerce"
    ),
    "availability": books_df["availability"].astype("string"),
    "source": "Books to Scrape"
})

# Chuẩn hóa dữ liệu Shopee
shopee_static = pd.DataFrame({
    "product_id": [
        f"SHOP{i:06d}" for i in range(1, len(shopee_df) + 1)
    ],
    "title": shopee_df["title"].astype("string").str.strip(),
    "price": pd.to_numeric(
        shopee_df["price_actual"].astype(str).str.replace(r"[^\d.]", "", regex=True),
        errors="coerce"
    ),
    "rating": pd.to_numeric(
        shopee_df["item_rating"], errors="coerce"
    ),
    "availability": pd.NA,
    "source": "Shopee"
})

print("Books standardized:", books_static.shape)
print("Shopee standardized:", shopee_static.shape)

display(books_static.head())
display(shopee_static.head())


Books standardized: (300, 6)
Shopee standardized: (20312, 6)


,product_id,title,price,rating,availability,source
0,BOOK000001,A Light in the Attic,51.77,NaN,In stock,Books to Scrape
1,BOOK000002,Tipping the Velvet,53.74,NaN,In stock,Books to Scrape
2,BOOK000003,Soumission,50.10,NaN,In stock,Books to Scrape
3,BOOK000004,Sharp Objects,47.82,NaN,In stock,Books to Scrape
4,BOOK000005,Sapiens: A Brief History of Humankind,54.23,NaN,In stock,Books to Scrape


,product_id,title,price,rating,availability,source
0,SHOP000001,1081 PU SLING BAG KULIT leather Casual Beg Tan...,12.90,4.9,<NA>,Shopee
1,SHOP000002,Stat Chinese Traditional Frog Buttons Knot Che...,0.75,5.0,<NA>,Shopee
2,SHOP000003,N09 READY STOCK MYFOOYIN woman handbag set 2in...,8.50,4.9,<NA>,Shopee
3,SHOP000004,🔥Shocking Sale🔥Cotton Comfort Nursing Bra Brea...,8.70,4.9,<NA>,Shopee
4,SHOP000005,Canon EOS 70D 18-55mm IS STM Original+64gb+ext...,2739.00,5.0,<NA>,Shopee


In [4]:
unified_static_df = pd.concat(
    [books_static, shopee_static],
    ignore_index=True
)

print("Unified Static Master Table:", unified_static_df.shape)
display(unified_static_df.head(10))

Unified Static Master Table: (20612, 6)


C:\Users\huynh\AppData\Local\Temp\ipykernel_25868\2706890969.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  unified_static_df = pd.concat(


,product_id,title,price,rating,availability,source
0,BOOK000001,A Light in the Attic,51.77,NaN,In stock,Books to Scrape
1,BOOK000002,Tipping the Velvet,53.74,NaN,In stock,Books to Scrape
2,BOOK000003,Soumission,50.10,NaN,In stock,Books to Scrape
3,BOOK000004,Sharp Objects,47.82,NaN,In stock,Books to Scrape
4,BOOK000005,Sapiens: A Brief History of Humankind,54.23,NaN,In stock,Books to Scrape
5,BOOK000006,The Requiem Red,22.65,NaN,In stock,Books to Scrape
6,BOOK000007,The Dirty Little Secrets of Getting Your Dream...,33.34,NaN,In stock,Books to Scrape
7,BOOK000008,The Coming Woman: A Novel Based on the Life of...,17.93,NaN,In stock,Books to Scrape
8,BOOK000009,The Boys in the Boat: Nine Americans and Their...,22.60,NaN,In stock,Books to Scrape
9,BOOK000010,The Black Maria,52.15,NaN,In stock,Books to Scrape


In [5]:
print("Số dòng:", len(unified_static_df))
print("Số cột:", len(unified_static_df.columns))

print("\nSố sản phẩm theo nguồn:")
print(unified_static_df["source"].value_counts())

print("\nMissing values:")
print(unified_static_df.isna().sum())

print("\nDuplicate product IDs:")
print(unified_static_df["product_id"].duplicated().sum())


Số dòng: 20612
Số cột: 6

Số sản phẩm theo nguồn:
source
Shopee             20312
Books to Scrape      300
Name: count, dtype: int64

Missing values:
product_id          0
title               0
price              15
rating           1228
availability    20312
source              0
dtype: int64

Duplicate product IDs:
0


In [6]:
STATIC_PATH = os.path.join(
    DATA_DIR,
    "unified_static_master.csv"
)

unified_static_df.to_csv(
    STATIC_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Đã lưu:", STATIC_PATH)


Đã lưu: D:\L-NMPTDL&AI\Stage 1 + 2\DA-AI_Nhom8\DHHTTT20B_TCTA_Nhom8_Stage1\data\unified_static_master.csv
